# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

In [12]:
# ============================================================
# ML-04 — Section 1: Unit of analysis + time window
# ============================================================

print("""
UNIT OF ANALYSIS

One row represents one report date × pseudonymized client ×
pseudonymized content item in fact_content_daily_performance.

The client and content identifiers are pseudonymized hashes.
The observed time window is measured directly from report_date.
""")

# ------------------------------------------------------------
# Verify row count, grain and date window
# ------------------------------------------------------------

grain_check = con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT report_date) AS report_dates,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT content_hash_id) AS content_items,
    MIN(report_date) AS min_report_date,
    MAX(report_date) AS max_report_date
FROM read_parquet('{DAILY}')
""")

display(grain_check)

# ------------------------------------------------------------
# Verify proposed grain
# ------------------------------------------------------------

duplicate_grain = con.sql(f"""
SELECT
    COUNT(*) AS duplicate_grain_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        COUNT(*) AS n
    FROM read_parquet('{DAILY}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
""")

display(duplicate_grain)

print("""
If duplicate_grain_groups = 0, the proposed unit of analysis
is supported by the observed data.
""")


UNIT OF ANALYSIS

One row represents one report date × pseudonymized client ×
pseudonymized content item in fact_content_daily_performance.

The client and content identifiers are pseudonymized hashes.
The observed time window is measured directly from report_date.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬──────────────┬─────────┬───────────────┬─────────────────┬─────────────────┐
│ row_count │ report_dates │ clients │ content_items │ min_report_date │ max_report_date │
│   int64   │    int64     │  int64  │     int64     │      date       │      date       │
├───────────┼──────────────┼─────────┼───────────────┼─────────────────┼─────────────────┤
│  78835655 │          520 │      70 │        427292 │ 2025-01-27      │ 2026-06-30      │
└───────────┴──────────────┴─────────┴───────────────┴─────────────────┴─────────────────┘

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────┐
│ duplicate_grain_groups │
│         int64          │
├────────────────────────┤
│                   6390 │
└────────────────────────┘


If duplicate_grain_groups = 0, the proposed unit of analysis
is supported by the observed data.



## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [13]:

FEATURES = [
    "content_age_days",
    "days_since_last_update",
    "impressions_90d",
    "avg_position",
    "ctr",
    "word_count",
]

LABEL = "is_declining"

CONTEXT = [
    "report_date",
    "client_id",
    "content_id",
]

EXCLUDED = {
    "health_score":
        "Derived score; it summarizes other signals and can leak the outcome.",

    "needs_ctr_fix":
        "Derived recommendation flag; it is downstream of performance signals.",

    "needs_engagement_fix":
        "Derived recommendation flag; it is downstream of performance signals.",

    "is_quick_win":
        "Derived decision flag; using it would leak a business rule.",

    "ai_opportunity":
        "Derived opportunity flag; not a raw predictive input.",

    "is_underperformer":
        "Derived performance flag; overlaps with the target/outcome.",

    "is_initial_refresh_candidate":
        "Derived recommendation flag; overlaps with the decision being modeled.",

    "is_declining":
        "Target/label, not a feature.",

    "trend_direction":
        "Directly represents the outcome used to define the decline label.",

    "trend_pct":
        "Direct outcome-derived measure; using it as a feature would leak the label.",

    "client_id":
        "Context identifier; do not use as a predictive feature.",

    "content_id":
        "Context identifier; do not use as a predictive feature.",
}

print("FEATURES")
for x in FEATURES:
    print(" -", x)

print("\nLABEL")
print(" -", LABEL)

print("\nCONTEXT")
for x in CONTEXT:
    print(" -", x)

print("\nEXCLUDED")
for field, reason in EXCLUDED.items():
    print(f" - {field}: {reason}")

schema = con.sql(f"""
DESCRIBE SELECT *
FROM read_parquet('{DAILY}')
""").df()

available_columns = set(schema["column_name"])

requested = set(FEATURES + [LABEL] + CONTEXT + list(EXCLUDED.keys()))

missing = sorted(requested - available_columns)

print("\nCOLUMN CHECK")
if missing:
    print("Missing columns:", missing)
else:
    print("All contract fields are present in the daily table.")

FEATURES
 - content_age_days
 - days_since_last_update
 - impressions_90d
 - avg_position
 - ctr
 - word_count

LABEL
 - is_declining

CONTEXT
 - report_date
 - client_id
 - content_id

EXCLUDED
 - health_score: Derived score; it summarizes other signals and can leak the outcome.
 - needs_ctr_fix: Derived recommendation flag; it is downstream of performance signals.
 - needs_engagement_fix: Derived recommendation flag; it is downstream of performance signals.
 - is_quick_win: Derived decision flag; using it would leak a business rule.
 - ai_opportunity: Derived opportunity flag; not a raw predictive input.
 - is_underperformer: Derived performance flag; overlaps with the target/outcome.
 - is_initial_refresh_candidate: Derived recommendation flag; overlaps with the decision being modeled.
 - is_declining: Target/label, not a feature.
 - trend_direction: Directly represents the outcome used to define the decline label.
 - trend_pct: Direct outcome-derived measure; using it as a feature 

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [10]:
schema = con.sql(f"""
DESCRIBE
SELECT *
FROM read_parquet('{DAILY}')
""").df()

display(schema)

,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None


In [15]:
# ============================================================
# 3A — COUNTS + WINDOW
# ============================================================

print("=== 3A — COUNTS + WINDOW ===")

con.sql(f"""
SELECT
    COUNT(*) AS row_count,
    COUNT(DISTINCT report_date) AS distinct_dates,
    COUNT(DISTINCT client_hash_id) AS distinct_clients,
    COUNT(DISTINCT content_hash_id) AS distinct_content_items,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date
FROM read_parquet('{DAILY}')
""").show()

=== 3A — COUNTS + WINDOW ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────────┬────────────────┬──────────────────┬────────────────────────┬────────────┬────────────┐
│ row_count │ distinct_dates │ distinct_clients │ distinct_content_items │ first_date │ last_date  │
│   int64   │     int64      │      int64       │         int64          │    date    │    date    │
├───────────┼────────────────┼──────────────────┼────────────────────────┼────────────┼────────────┤
│  78835655 │            520 │               70 │                 427292 │ 2025-01-27 │ 2026-06-30 │
└───────────┴────────────────┴──────────────────┴────────────────────────┴────────────┴────────────┘



In [16]:
print("=== 3B — GRAIN CHECK ===")

con.sql(f"""
SELECT
    COUNT(*) AS duplicate_grain_groups
FROM (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id
    FROM read_parquet('{DAILY}')
    GROUP BY
        report_date,
        client_hash_id,
        content_hash_id
    HAVING COUNT(*) > 1
)
""").show()

=== 3B — GRAIN CHECK ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌────────────────────────┐
│ duplicate_grain_groups │
│         int64          │
├────────────────────────┤
│                   6390 │
└────────────────────────┘



## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [18]:
# ============================================================
# ML-04 — Section 4: Data limits
# ============================================================

print("""
DATA LIMITS

1. This is observational search-performance data.
   It can measure patterns and associations in the observed data,
   but it cannot prove that a specific change caused a ranking
   or traffic change.

2. The client history is not necessarily balanced.
   Different clients can have different amounts of observed history,
   so comparisons across clients should be treated carefully.

3. Early observations can have limited Google Search Console history.
   Missing or shorter history should not automatically be treated
   as zero performance.

4. The query data uses 90-day windows.
   Last-30-day and previous-30-day measurements are window-based,
   so they should not be treated as independent daily observations.

5. Window-based observations can overlap across reporting periods.
   Adjacent observations may therefore share some underlying data.

6. The data is anonymized/pseudonymized.
   It does not provide client names, URLs, titles, or raw search
   queries, so the analysis cannot explain specific real-world
   brands, pages, or keywords.

7. The data supports observed, measured, directional,
   decision-support analysis.
   It cannot tell us Google's ranking algorithm or prove why
   Google changed a page's ranking.

8. Derived recommendation or outcome fields should not be used
   as predictive features when they contain information derived
   from the same performance signals or target outcome.
""")

# ------------------------------------------------------------
# Verify client history depth
# ------------------------------------------------------------

print("=== CLIENT HISTORY DEPTH CHECK ===")

client_history = con.sql(f"""
SELECT
    client_hash_id,
    MIN(report_date) AS first_date,
    MAX(report_date) AS last_date,
    COUNT(DISTINCT report_date) AS observed_days
FROM read_parquet('{DAILY}')
GROUP BY client_hash_id
ORDER BY observed_days
""").df()

display(client_history)

print(
    "Minimum observed client-days:",
    client_history["observed_days"].min()
)

print(
    "Maximum observed client-days:",
    client_history["observed_days"].max()
)

# ------------------------------------------------------------
# Verify client/date coverage
# ------------------------------------------------------------

print("=== CLIENT-DATE COVERAGE CHECK ===")

coverage = con.sql(f"""
SELECT
    COUNT(*) AS observed_client_date_pairs,
    COUNT(DISTINCT client_hash_id) AS clients,
    COUNT(DISTINCT report_date) AS dates
FROM (
    SELECT DISTINCT
        client_hash_id,
        report_date
    FROM read_parquet('{DAILY}')
)
""").df()

display(coverage)

# ------------------------------------------------------------
# Calculate the theoretical full balanced-panel size
# ------------------------------------------------------------

coverage_values = coverage.iloc[0]

observed_pairs = int(coverage_values["observed_client_date_pairs"])
clients = int(coverage_values["clients"])
dates = int(coverage_values["dates"])

possible_pairs = clients * dates

print("Possible client × date pairs:", possible_pairs)
print("Observed client × date pairs:", observed_pairs)
print("Missing client × date pairs:", possible_pairs - observed_pairs)

if observed_pairs < possible_pairs:
    print(
        "\nConclusion: the observed panel is not fully balanced. "
        "Not every client has an observation for every report date."
    )
else:
    print(
        "\nConclusion: every client has an observation for every "
        "report date in the observed window."
    )


DATA LIMITS

1. This is observational search-performance data.
   It can measure patterns and associations in the observed data,
   but it cannot prove that a specific change caused a ranking
   or traffic change.

2. The client history is not necessarily balanced.
   Different clients can have different amounts of observed history,
   so comparisons across clients should be treated carefully.

3. Early observations can have limited Google Search Console history.
   Missing or shorter history should not automatically be treated
   as zero performance.

4. The query data uses 90-day windows.
   Last-30-day and previous-30-day measurements are window-based,
   so they should not be treated as independent daily observations.

5. Window-based observations can overlap across reporting periods.
   Adjacent observations may therefore share some underlying data.

6. The data is anonymized/pseudonymized.
   It does not provide client names, URLs, titles, or raw search
   queries, so the analys

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,first_date,last_date,observed_days
0,client_aef6ffea193da149,2026-05-28,2026-06-30,34
1,client_04660893ae39614a,2026-05-22,2026-06-30,40
2,client_a22068e339bf95f5,2026-05-22,2026-06-30,40
3,client_c353557474475e51,2026-04-30,2026-06-30,52
4,client_1a8bf67cad4ee525,2026-05-09,2026-06-30,53
...,...,...,...,...
65,client_62f4a7e64f5e0096,2025-06-07,2026-06-30,389
66,client_fef1a8f436438636,2025-03-11,2026-06-30,477
67,client_73cda7b4e4f265ea,2025-02-11,2026-06-30,505
68,client_ff644d8251367cbb,2025-01-27,2026-06-30,514


Minimum observed client-days: 34
Maximum observed client-days: 520
=== CLIENT-DATE COVERAGE CHECK ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,observed_client_date_pairs,clients,dates
0,15069,70,520


Possible client × date pairs: 36400
Observed client × date pairs: 15069
Missing client × date pairs: 21331

Conclusion: the observed panel is not fully balanced. Not every client has an observation for every report date.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.